In [1]:
import extract
from ddl import run_ddl, drop_all_tables
from initial_load import run_initial_load
from datamart import populate_datamart
import pandas as pd
from sqlalchemy import create_engine
from config import (
    DB_HOST,
    DB_PORT,
    DB_NAME,
    DB_USER,
    DB_PASSWORD,
    DWH_SCHEMA,
    DM_SCHEMA,
    FACT_SALES,
    DIM_CUSTOMER,
    DIM_PRODUCT,
    DIM_STORE,
    DIM_DATE,
    DM_SUMMARY_SALES_PER_HOUR,
    DM_SUMMARY_PRODUCT_SALES,
    DM_SUMMARY_STORE_SALES,
    DM_SUMMARY_DAILY_SALES
)

In [2]:
df = extract.extract_data()

df

,sales_id,transaction_number,sales_date,store_id,store_city,customer_id,first_name,last_name,product_id,product_name,category_name,quantity,price,discount,sales
0,1,FQL4S94E4ME1EZFTG42G,2018-02-05 07:38:25.43,3,Tangerang,27039,Susan,Green,381,Vaccum Bag 10x13,Confections,7,44.2337,0.0,309.63590
1,2,12UGLX40DJ1A5DTFBHB8,2018-02-02 16:03:31.15,6,Denpasar,25011,Telly,Pollard,61,Sardines,Grain,7,62.5460,0.0,437.82200
2,3,5DT8RCPL87KI5EORO7B0,2018-05-03 19:31:56.88,3,Tangerang,94024,Jon,Rangel,23,Crab - Imitation Flakes,Produce,24,79.0184,0.0,1896.44160
3,4,R3DR9MLD5NR76VO17ULE,2018-04-07 14:43:55.42,2,Bogor,73966,Carol,Gilmore,176,Smirnoff Green Apple Twist,Seafood,19,81.3167,0.2,1236.01384
4,5,4BGS0Z5OMAZ8NDAFHHP3,2018-02-12 15:37:03.94,2,Bogor,32653,Terra,Carter,310,Coffee - Dark Roast,Poultry,9,79.9780,0.0,719.80200
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59419,59996,UGYR2FS3LNF1ZP63AEEJ,2018-03-14 18:33:02.69,5,Bandung,36899,Denise,Frederick,322,Onions - Vidalia,Shell fish,10,26.2846,0.0,262.84600
59420,59997,N3BMRNWW9ZYBEEUKGSQC,2018-02-25 17:24:23.84,1,Jakarta,92487,Teri,Anderson,133,Beef - Top Sirloin - Aaa,Produce,24,2.9327,0.0,70.38480
59421,59998,IHY5QMD07OFU37CN5ASV,2018-04-14 18:57:39.05,3,Tangerang,60783,Elena,Fry,334,Ecolab - Lime - A - Way 4/4 L,Meat,16,21.6133,0.0,345.81280
59422,59999,COGQKBBJUMN7J5MCH10J,2018-01-09 08:40:02.66,2,Bogor,85118,Leo,Garcia,188,Cup - Translucent 7 Oz Clear,Grain,22,78.1768,0.0,1719.88960


In [2]:
import sys
print(sys.executable)

d:\Anaconda\python.exe


In [4]:
drop_all_tables(confirm=True)
run_ddl()

Semua tabel berhasil di-drop.
Schemas created successfully.
DWH tables created successfully.
Daily partitions created from 2018-01-01 to 2018-05-10.
Datamart tables created successfully.
=== DDL COMPLETED ===


In [5]:
# proses transform

import importlib
import transform

importlib.reload(transform)

dim_customer = transform.transform_dim_customer(df)
dim_product = transform.transform_dim_product(df)
dim_store = transform.transform_dim_store(df)
dim_date = transform.transform_dim_date(df)
fact_sales = transform.transform_fact_sales(df)

In [6]:
# Process load

from initial_load import run_initial_load
run_initial_load()

Starting Initial Load...
RAW: (59424, 15)
   sales_id    transaction_number              sales_date  store_id  \
0         1  FQL4S94E4ME1EZFTG42G  2018-02-05 07:38:25.43         3   
1         2  12UGLX40DJ1A5DTFBHB8  2018-02-02 16:03:31.15         6   
2         3  5DT8RCPL87KI5EORO7B0  2018-05-03 19:31:56.88         3   
3         4  R3DR9MLD5NR76VO17ULE  2018-04-07 14:43:55.42         2   
4         5  4BGS0Z5OMAZ8NDAFHHP3  2018-02-12 15:37:03.94         2   

  store_city  customer_id first_name last_name  product_id  \
0  Tangerang        27039      Susan     Green         381   
1   Denpasar        25011      Telly   Pollard          61   
2  Tangerang        94024        Jon    Rangel          23   
3      Bogor        73966      Carol   Gilmore         176   
4      Bogor        32653      Terra    Carter         310   

                 product_name category_name  quantity    price  discount  \
0            Vaccum Bag 10x13   Confections         7  44.2337       0.0   
1     

In [7]:
import psycopg2
from config import DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD

conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
)

cur = conn.cursor()

cur.execute("""
SELECT schema_name
FROM information_schema.schemata
WHERE schema_name IN ('dwh', 'datamart');
""")

schemas = cur.fetchall()

for s in schemas:
    print(s[0])

cur.close()
conn.close()

datamart
dwh


In [8]:
import psycopg2
from config import DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD

conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
)

cur = conn.cursor()

cur.execute("""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema = 'dwh'
ORDER BY table_name;
""")

tables = cur.fetchall()

for t in tables:
    print(t)

cur.close()
conn.close()

('dwh', 'dim_customer')
('dwh', 'dim_date')
('dwh', 'dim_product')
('dwh', 'dim_store')
('dwh', 'fact_sales')
('dwh', 'fact_sales_20180101')
('dwh', 'fact_sales_20180102')
('dwh', 'fact_sales_20180103')
('dwh', 'fact_sales_20180104')
('dwh', 'fact_sales_20180105')
('dwh', 'fact_sales_20180106')
('dwh', 'fact_sales_20180107')
('dwh', 'fact_sales_20180108')
('dwh', 'fact_sales_20180109')
('dwh', 'fact_sales_20180110')
('dwh', 'fact_sales_20180111')
('dwh', 'fact_sales_20180112')
('dwh', 'fact_sales_20180113')
('dwh', 'fact_sales_20180114')
('dwh', 'fact_sales_20180115')
('dwh', 'fact_sales_20180116')
('dwh', 'fact_sales_20180117')
('dwh', 'fact_sales_20180118')
('dwh', 'fact_sales_20180119')
('dwh', 'fact_sales_20180120')
('dwh', 'fact_sales_20180121')
('dwh', 'fact_sales_20180122')
('dwh', 'fact_sales_20180123')
('dwh', 'fact_sales_20180124')
('dwh', 'fact_sales_20180125')
('dwh', 'fact_sales_20180126')
('dwh', 'fact_sales_20180127')
('dwh', 'fact_sales_20180128')
('dwh', 'fact_sales_20

In [9]:
import psycopg2
from config import DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD

conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
)

cur = conn.cursor()

cur.execute("""
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema = 'datamart'
ORDER BY table_name;
""")

tables = cur.fetchall()

for t in tables:
    print(t)

cur.close()
conn.close()

('datamart', 'summary_daily_sales')
('datamart', 'summary_product_sales')
('datamart', 'summary_sales_per_hour')
('datamart', 'summary_store_sales')


In [10]:
import psycopg2
from config import DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD

conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
)

cur = conn.cursor()

cur.execute("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'dwh'
  AND table_name = 'fact_sales'
ORDER BY ordinal_position;
""")

columns = cur.fetchall()

for c in columns:
    print(c)

cur.close()
conn.close()

('sales_id', 'integer')
('transaction_number', 'character varying')
('sales_date', 'timestamp without time zone')
('store_id', 'integer')
('customer_id', 'integer')
('product_id', 'integer')
('price', 'numeric')
('quantity', 'integer')
('discount', 'numeric')
('sales', 'numeric')


In [11]:
dim_product

,product_id,product_name,product_category
0,381,Vaccum Bag 10x13,Confections
1,61,Sardines,Grain
2,23,Crab - Imitation Flakes,Produce
3,176,Smirnoff Green Apple Twist,Seafood
4,310,Coffee - Dark Roast,Poultry
...,...,...,...
1886,88,Remy Red,Shell fish
2602,301,Beer - Labatt Blue,Grain
2692,268,Vanilla Beans,Poultry
2840,430,Milk - 1%,Shell fish


In [12]:
import pandas as pd
from sqlalchemy import create_engine
from config import DB_HOST, DB_PORT, DB_NAME, DB_USER, DB_PASSWORD

# connection string
conn_str = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(conn_str)

# query
product_ids = 392
query = f"SELECT * FROM dwh.dim_product WHERE product_id = {product_ids}"

df_product_ids = pd.read_sql(query, engine)
print(df)

       sales_id    transaction_number              sales_date  store_id  \
0             1  FQL4S94E4ME1EZFTG42G  2018-02-05 07:38:25.43         3   
1             2  12UGLX40DJ1A5DTFBHB8  2018-02-02 16:03:31.15         6   
2             3  5DT8RCPL87KI5EORO7B0  2018-05-03 19:31:56.88         3   
3             4  R3DR9MLD5NR76VO17ULE  2018-04-07 14:43:55.42         2   
4             5  4BGS0Z5OMAZ8NDAFHHP3  2018-02-12 15:37:03.94         2   
...         ...                   ...                     ...       ...   
59419     59996  UGYR2FS3LNF1ZP63AEEJ  2018-03-14 18:33:02.69         5   
59420     59997  N3BMRNWW9ZYBEEUKGSQC  2018-02-25 17:24:23.84         1   
59421     59998  IHY5QMD07OFU37CN5ASV  2018-04-14 18:57:39.05         3   
59422     59999  COGQKBBJUMN7J5MCH10J  2018-01-09 08:40:02.66         2   
59423     60000  2AIGB7HWAHYHW6SHE5B1  2018-01-10 17:26:23.88         6   

      store_city  customer_id first_name  last_name  product_id  \
0      Tangerang        27039   

In [13]:
dim_customer

,customer_id,first_name,last_name
0,27039,Susan,Green
1,25011,Telly,Pollard
2,94024,Jon,Rangel
3,73966,Carol,Gilmore
4,32653,Terra,Carter
...,...,...,...
59414,45271,Alisa,Acosta
59417,4481,Leroy,Henson
59420,92487,Teri,Anderson
59421,60783,Elena,Fry


In [14]:
dim_date.head(5)

,sales_date,year,month,day,hour
0,2018-02-05 07:38:25.430,2018,2,5,7
1,2018-02-02 16:03:31.150,2018,2,2,16
2,2018-05-03 19:31:56.880,2018,5,3,19
3,2018-04-07 14:43:55.420,2018,4,7,14
4,2018-02-12 15:37:03.940,2018,2,12,15


In [15]:
# run datamart

from datamart import populate_datamart

populate_datamart()

Datamart population completed!


In [16]:
dim_store

,store_id,store_city
0,3,Tangerang
1,6,Denpasar
3,2,Bogor
7,5,Bandung
8,4,Depok
11,1,Jakarta


In [17]:
query = "SELECT COUNT(*) AS total FROM dwh.dim_customer"
df_check = pd.read_sql(query, engine)
print(df_check)

   total
0  44574


In [18]:
print("dim_customer shape:", dim_customer.shape)
print(dim_customer.head())

dim_customer shape: (44574, 3)
   customer_id first_name last_name
0        27039      Susan     Green
1        25011      Telly   Pollard
2        94024        Jon    Rangel
3        73966      Carol   Gilmore
4        32653      Terra    Carter


In [19]:
from load import get_connection

conn = get_connection()
cur = conn.cursor()

cur.execute("SELECT current_database(), current_schema();")
print(cur.fetchall())

cur.close()
conn.close()

[('grocery_dw', 'public')]
